# 03 - Privacy Demonstration and Policy Recommendation

### Table of contents

0. [Setup and Data Loading](#0-setup-and-data-loading)
1. [Identification of Personal Identifiable Information (PII)](#1-identification-of-personal-identifiable-information-pii)
2. [Uniqueness Analysis of Quasi Identifiers](#2-uniqueness-analysis-of-quasi-identifiers)
3. [Pseudonymization and removal of Direct Identifiers](#3-pseudonymization-and-removal-of-direct-identifiers)
4. [GDPR Compliance Mapping](#4-gdpr-compliance-mapping)
5. [Privacy and Governance Gaps](#5-privacy-and-governance-gaps)
6. [Governance Recommendations](#6-governance-recommendations)

## 0. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
from scipy import stats
import json

# visual style
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
pallete_colors = {'Male': '#E67E22', 'Female': '#9B59B6', 'Unknown': '#8c8c8c'}

In [2]:
# Data loading and exploration
df = pd.read_json("../data/raw/raw_credit_applications.json")

print(df.head())
print(df.info())

       _id                                     applicant_info  \
0  app_200  {'full_name': 'Jerry Smith', 'email': 'jerry.s...   
1  app_037  {'full_name': 'Brandon Walker', 'email': 'bran...   
2  app_215  {'full_name': 'Scott Moore', 'email': 'scott.m...   
3  app_024  {'full_name': 'Thomas Lee', 'email': 'thomas.l...   
4  app_184  {'full_name': 'Brian Rodriguez', 'email': 'bri...   

                                          financials  \
0  {'annual_income': 73000, 'credit_history_month...   
1  {'annual_income': 78000, 'credit_history_month...   
2  {'annual_income': 61000, 'credit_history_month...   
3  {'annual_income': 103000, 'credit_history_mont...   
4  {'annual_income': 57000, 'credit_history_month...   

                                   spending_behavior  \
0  [{'category': 'Shopping', 'amount': 480}, {'ca...   
1  [{'category': 'Rent', 'amount': 608}, {'catego...   
2              [{'category': 'Rent', 'amount': 109}]   
3           [{'category': 'Fitness', 'amount': 5

## 1. Identification of Personal Identifiable Information (PII)

### 1.1 Direct Identifiers

Direct identifiers are variables that allow to immediate identication of an individual without the need to combine any other data.

This datast contains the following **Direct Identifiers** :

* **applicant_info.full_name**: This field contains the full legal name of each applicant. A full name directly identifies a specific individual.

* **applicant_info.email**: Email addresses are unique to each individual and allow for direct contact. 

* **applicant_info.ssn**: The Social Security Number is a unique national identifier, directly identifiying the person to whom it belongs to.

* **applicant_info.ip_address**: An IP address is assigned to a singular device connected to the internet. While it doesn't directly reveal the persons identity, it can be linked to a single individual through an internet provider, for example.

### 1.2 Quasi-Identifiers

Quasi-identifiers cannot identify a specific individual on their own. However, when combined with other attributes, they can significantly narrow down the search and potentially lead to the identification of a single individual.

This dataset contains the following **Quasi-Identifiers**:

* **applicant_info.gender**: A person’s gender alone cannot reveal their identity. However, when combined with other attributes such as date of birth or zip code, it can substantially reduce the number of possible matches, especially in smaller datasets.

* **application_info.date_of_birth**: Many people share the same birthday, so this field alone does not identify an individual. However, when combined with other variables such as zip code and gender, it can drastically narrow down the number of possible individuals. In many cases, only one person may match a specific combination.

* **application_info.zip_code**: A zip code does not directly identify a person, but it narrows the search to a specific geographic location. When paired with attributes such as date of birth and gender, it can significantly increase the likelihood of identifying a single individual.

## 2. Uniqueness Analysis of Quasi-Identifiers

### 2.1 Checking Possible individual Identification Through Quasi-Identifier combinations

In [3]:
# Quasi-Identifiers columns

df["date_of_birth"] = df["applicant_info"].apply(lambda x: x.get("date_of_birth") if isinstance(x, dict) else None)
df["gender"]        = df["applicant_info"].apply(lambda x: x.get("gender") if isinstance(x, dict) else None)
df["zip_code"]      = df["applicant_info"].apply(lambda x: x.get("zip_code") if isinstance(x, dict) else None)

#### 2.1.1 Checking for single variables 

In [4]:

# date_of_birth alone
groups = df.groupby(["date_of_birth"]).size().reset_index(name="count")
print("date_of_birth unique:", (groups["count"] == 1).sum())

# zip_code alone
groups = df.groupby(["zip_code"]).size().reset_index(name="count")
print("zip_code unique:", (groups["count"] == 1).sum())

# gender alone
groups = df.groupby(["gender"]).size().reset_index(name="count")
print("gender unique:", (groups["count"] == 1).sum())

date_of_birth unique: 489
zip_code unique: 51
gender unique: 0


The analysis shows that date_of_birth alone uniquely identifies 489 out of 502 individuals. This means most applicants have a date of birth that appears only once in the dataset, which creates a high re identification risk.

The variable zip_code uniquely identifies 51 individuals. Although the risk is lower than for date_of_birth, it still allows a meaningful number of applicants to be singled out.

The variable gender does not uniquely identify any individual. This is expected because gender has a very small number of categories, so many applicants share the same value.

#### 2.1.2 Checking for combinations between 2 variables

In [5]:
# date_of_birth + zip_code
groups = df.groupby(["date_of_birth", "zip_code"]).size().reset_index(name="count")
print("dob + zip unique:", (groups["count"] == 1).sum())

# date_of_birth + gender
groups = df.groupby(["date_of_birth", "gender"]).size().reset_index(name="count")
print("dob + gender unique:", (groups["count"] == 1).sum())

# zip_code + gender
groups = df.groupby(["zip_code", "gender"]).size().reset_index(name="count")
print("zip + gender unique:", (groups["count"] == 1).sum())

dob + zip unique: 499
dob + gender unique: 495
zip + gender unique: 181


The combination of date_of_birth and zip_code singles out 499 out of 502 individuals. This means almost the entirety of the dataset can be uniquely identified when these two variables are used together.

The combination of date_of_birth and gender uniquely identifies 495 individuals. Again, date_of_birth drives most of the uniqueness, and the addition of gender further increases the level of singularity.

The combination of zip_code and gender uniquely identifies 181 individuals. Although significantly less than the other two combinations, it still uniquely identifies an important proportion of the dataset.

In [6]:
groups = df.groupby(["date_of_birth", "zip_code", "gender"]).size().reset_index(name="count")
print("dob + zip + gender unique:", (groups["count"] == 1).sum())

dob + zip + gender unique: 499


When we combine the three variables we can identify 499 people out of 502, representing almost the entire dataset. Previous data quality analysis identified duplicated records, which may explain the small number of non unique cases.

## 3. Pseudonymization and removal of Direct Identifiers

### 3.1. Hashing direct Identifiers

In [7]:
df_before = df.copy()

def hash_value(value):
    if value is None:
        return None
    return hashlib.sha256(str(value).encode()).hexdigest()[:10]

df["applicant_info"] = df["applicant_info"].apply(
    lambda x: {**x,
               "full_name": hash_value(x.get("full_name")),
               "email": hash_value(x.get("email")),
               "ssn": hash_value(x.get("ssn")),
               "ip_address": hash_value(x.get("ip_address"))}
)

print("Before hashing")

for i in range(3):
    info = df_before["applicant_info"].iloc[i]
    print({
        "full_name": info.get("full_name"),
        "email": info.get("email"),
        "ssn": info.get("ssn"),
        "ip_address": info.get("ip_address")
    })

print("\nAfter hashing")

for i in range(3):
    info = df["applicant_info"].iloc[i]
    print({
        "full_name": info.get("full_name"),
        "email": info.get("email"),
        "ssn": info.get("ssn"),
        "ip_address": info.get("ip_address")
    })

Before hashing
{'full_name': 'Jerry Smith', 'email': 'jerry.smith17@hotmail.com', 'ssn': '596-64-4340', 'ip_address': '192.168.48.155'}
{'full_name': 'Brandon Walker', 'email': 'brandon.walker2@yahoo.com', 'ssn': '425-69-4784', 'ip_address': '10.1.102.112'}
{'full_name': 'Scott Moore', 'email': 'scott.moore94@mail.com', 'ssn': '370-78-5178', 'ip_address': '10.240.193.250'}

After hashing
{'full_name': '68ee17cf46', 'email': '116648a776', 'ssn': '2caf30528c', 'ip_address': 'afb811b70b'}
{'full_name': '4c539f3c4c', 'email': 'c3522c0b54', 'ssn': '2f7da45fef', 'ip_address': 'f661245c5f'}
{'full_name': '4ad1a6eb65', 'email': 'b299e7d6a3', 'ssn': 'db120edcee', 'ip_address': '21d90a6a0a'}


### 3.2 Hashing Hidden Trap: Brute force reversall

The hashing applied above appears secure, but a weakness exists. Hashing operates deterministically. The same input always produces the same hash output.

This property allows an attacker to generate possible input values and compare their hashes with the values stored in the dataset. For example, an attacker generates many possible IP addresses, computes the hash for each candidate, and compares the result with the stored hashes.When a matching hash appears, the attacker identifies the original IP addres

In [8]:
original_ip = "192.168.48.155"

# create target hash
target_hash = hashlib.sha256(original_ip.encode()).hexdigest()

print("target hash:", target_hash)

# simulate attacker testing possible IP values
for i in range(150, 160):
    candidate = f"192.168.48.{i}"
    candidate_hash = hashlib.sha256(candidate.encode()).hexdigest()

    if candidate_hash == target_hash:
        print("match found:", candidate)
        break

target hash: afb811b70bdc95ef34b9d872ee8a83a19525df6246d250dcb62f0d9fb93f2c72
match found: 192.168.48.155


### 3.3 The Fix: Adding a Salt

A salt is a random secret string appended to a value before hashing. The hash is computed from the combination between the salt and the original value, this makes attackers not able to reproduce the same hash even if they kknow the original input.

In [9]:
import secrets
# Step 1: generate secure salt
salt = secrets.token_hex(16)

email = "jerry.smith17@hotmail.com"

# Step 2: create salted hash
salted_hash = hashlib.sha256((salt + email).encode()).hexdigest()

print("salt (stored separately):", salt)
print("salted hash:", salted_hash)

# Step 3: simulate attacker without salt
attacker_attempt = hashlib.sha256(email.encode()).hexdigest()

print("\nattacker hash without salt:", attacker_attempt)
print("real salted hash:", salted_hash)

print("\nmatch:", attacker_attempt == salted_hash)

salt (stored separately): 741a0c5b4fb3c14912453309e049a9c3
salted hash: 08621eb44a3eefaea3cc82efc387859f9c0e18fc732c6ad6fcdcef318271b931

attacker hash without salt: 116648a7761525746032d0ab323ceb01f50d11f793516437a5cf80804fc6fb9d
real salted hash: 08621eb44a3eefaea3cc82efc387859f9c0e18fc732c6ad6fcdcef318271b931

match: False


## 4. Data  Quality Audit

The audit evaluates how the raw dataset performs across the main data quality dimensions:
* **Accuracy**
* **Completness**
* **Consistency**
* **Uniqueness**
* **Validity**
* **Timeliness**

In [10]:
# dataset used for the audit
audit_dataset = df_before.copy()

print("Total records in raw dataset:", len(audit_dataset))

print("\nRecord attributes:")
for i, col in enumerate(audit_dataset.columns, 1):
    print(f"{i}. {col}")

print("\nNested structure:")

sample_row = audit_dataset.iloc[0]

for col in ["applicant_info", "financials", "decision", "spending_behavior"]:
    value = sample_row[col]

    if isinstance(value, dict):
        print(f"{col} fields:", list(value.keys()))

    elif isinstance(value, list) and len(value) > 0 and isinstance(value[0], dict):
        print(f"{col} fields:", list(value[0].keys()))

Total records in raw dataset: 502

Record attributes:
1. _id
2. applicant_info
3. financials
4. spending_behavior
5. decision
6. processing_timestamp
7. loan_purpose
8. notes
9. date_of_birth
10. gender
11. zip_code

Nested structure:
applicant_info fields: ['full_name', 'email', 'ssn', 'ip_address', 'gender', 'date_of_birth', 'zip_code']
financials fields: ['annual_income', 'credit_history_months', 'debt_to_income', 'savings_balance']
decision fields: ['loan_approved', 'rejection_reason']
spending_behavior fields: ['category', 'amount']


In [11]:
audit_dataset["full_name"] = audit_dataset["applicant_info"].apply(lambda x: x.get("full_name") if isinstance(x, dict) else None)
audit_dataset["email"] = audit_dataset["applicant_info"].apply(lambda x: x.get("email") if isinstance(x, dict) else None)
audit_dataset["ssn"] = audit_dataset["applicant_info"].apply(lambda x: x.get("ssn") if isinstance(x, dict) else None)
audit_dataset["ip_address"] = audit_dataset["applicant_info"].apply(lambda x: x.get("ip_address") if isinstance(x, dict) else None)
audit_dataset["gender"] = audit_dataset["applicant_info"].apply(lambda x: x.get("gender") if isinstance(x, dict) else None)
audit_dataset["date_of_birth"] = audit_dataset["applicant_info"].apply(lambda x: x.get("date_of_birth") if isinstance(x, dict) else None)
audit_dataset["zip_code"] = audit_dataset["applicant_info"].apply(lambda x: x.get("zip_code") if isinstance(x, dict) else None)

audit_dataset["annual_income"] = audit_dataset["financials"].apply(lambda x: x.get("annual_income") if isinstance(x, dict) else None)
audit_dataset["credit_history_months"] = audit_dataset["financials"].apply(lambda x: x.get("credit_history_months") if isinstance(x, dict) else None)
audit_dataset["debt_to_income"] = audit_dataset["financials"].apply(lambda x: x.get("debt_to_income") if isinstance(x, dict) else None)
audit_dataset["savings_balance"] = audit_dataset["financials"].apply(lambda x: x.get("savings_balance") if isinstance(x, dict) else None)

audit_dataset["loan_approved"] = audit_dataset["decision"].apply(lambda x: x.get("loan_approved") if isinstance(x, dict) else None)
audit_dataset["interest_rate"] = audit_dataset["decision"].apply(lambda x: x.get("interest_rate") if isinstance(x, dict) else None)
audit_dataset["approved_amount"] = audit_dataset["decision"].apply(lambda x: x.get("approved_amount") if isinstance(x, dict) else None)
audit_dataset["rejection_reason"] = audit_dataset["decision"].apply(lambda x: x.get("rejection_reason") if isinstance(x, dict) else None)

### 4.1 Audit Query 1: Find Duplicates (Uniqueness)

Data Quality Dimension: Uniqueness

In [12]:
# Check for matches in Ids

# find duplicate ids
duplicate_rows = audit_dataset[audit_dataset["_id"].duplicated(keep=False)]

duplicate_ids_df = pd.DataFrame(
    duplicate_rows[["_id", "full_name", "ssn"]]
    .sort_values("_id")
    .reset_index(drop=True)
)
print("Duplicate Application IDs\n")
print(duplicate_ids_df)


if len(duplicate_ids_df) > 0:
    print(f"\nWarning: Found {len(duplicate_ids_df)} records with duplicate Application IDs.")
else:
    print("\nNo duplicate Application IDs found.")

Duplicate Application IDs

       _id         full_name          ssn
0  app_001  Stephanie Nguyen  427-90-1892
1  app_001  Stephanie Nguyen         None
2  app_042      Joseph Lopez  652-70-5530
3  app_042      Joseph Lopez  652-70-5530



In [13]:
# find rows where SSN appears more than once (ignore missing values)
duplicate_ssn_rows = audit_dataset[
    audit_dataset["ssn"].notna() &
    audit_dataset["ssn"].duplicated(keep=False)
]

duplicate_ssn_df = pd.DataFrame(
    duplicate_ssn_rows[["ssn", "full_name", "_id"]]
    .sort_values(["ssn", "_id"])
)

print(duplicate_ssn_df.to_string(index=False))

if len(duplicate_ssn_df) > 0:
    print(f"\nWarning: Found {len(duplicate_ssn_df)} records with duplicate SSNs.")
else:
    print("\nNo duplicate SSNs found.")

        ssn      full_name     _id
652-70-5530   Joseph Lopez app_042
652-70-5530   Joseph Lopez app_042
780-24-9300    Gary Wilson app_016
780-24-9300 Susan Martinez app_088
937-72-8731   Sandra Smith app_101
937-72-8731    Samuel Hill app_234



**Takeaway:**

 The Uniqueness check fails. Duplicate values exist at both ID and SSN levels.

### 4.2 Audit Query 2: Check Consistency

Data Quality Dimension: Consistency

In [15]:
# Checking consistency for Gender values

# view all gender values and counts
gender_counts = audit_dataset["gender"].value_counts(dropna=False)

print("Gender value distribution:")
print(gender_counts)


Gender value distribution:
gender
Male      195
Female    193
F          58
M          53
            2
None        1
Name: count, dtype: int64


Six distinct values appear instead of a consistent set.
‘Male’ and ‘M’, and ‘Female’ and ‘F’, represent the same category but use different formats.
Two empty strings and one missing value also appear, which adds further inconsistency.

In [17]:
# Check consistency for date_of_birth values

import re

# patterns
pattern_ymd_dash = r"^\d{4}-\d{2}-\d{2}$"
pattern_dmy_dash = r"^\d{2}-\d{2}-\d{4}$"
pattern_ymd_slash = r"^\d{4}/\d{2}/\d{2}$"
pattern_dmy_slash = r"^\d{2}/\d{2}/\d{4}$"
pattern_mdy_slash = r"^\d{2}/\d{2}/\d{4}$"

# classify format
def detect_format(d):
    if d is None:
        return "missing"
    
    if re.match(pattern_ymd_dash, d):
        return "YYYY-MM-DD"
    
    if re.match(pattern_dmy_dash, d):
        return "DD-MM-YYYY"
    
    if re.match(pattern_ymd_slash, d):
        return "YYYY/MM/DD"
    
    if re.match(pattern_dmy_slash, d):
        return "DD/MM/YYYY"
    
    if re.match(pattern_mdy_slash, d):
        return "MM/DD/YYYY"
    
    return "other"

audit_dataset["dob_format"] = audit_dataset["date_of_birth"].apply(detect_format)

print("Date format distribution:")
print(audit_dataset["dob_format"].value_counts())

Date format distribution:
dob_format
YYYY-MM-DD    340
DD/MM/YYYY    101
YYYY/MM/DD     56
other           4
missing         1
Name: count, dtype: int64


There are 340 records using the YYYY-MM-DD format.
Another 101 records use the DD/MM/YYYY format, and 56 records use YYYY/MM/DD.
Four values do not match any of the detected patterns, and one value is missing.
These differences show the dataset does not follow a single standardized date format.

### 4.3  Audit Query 3: Find Missing Fields (Completeness)

Data Quality Dimension: Completness

In [20]:

# fields to audit
fields = [
"full_name",
"ssn",
"gender",
"date_of_birth",
"zip_code",
"ip_address",
"annual_income",
"processing_timestamp",
"loan_purpose",
"notes"
]
total_records = len(audit_dataset)

missing_summary = []

for field in fields:
    missing = audit_dataset[field].isna().sum() + (audit_dataset[field] == "").sum()
    present = total_records - missing
    percent = (missing / total_records) * 100
    missing_summary.append([field, missing, present, round(percent,2)])

missing_df = pd.DataFrame(
    missing_summary,
    columns=["field","missing_count","present_count","missing_percent"]
).sort_values("missing_percent", ascending=False)

print("Missing values by field:")
print(missing_df.to_string(index=False))

Missing values by field:
               field  missing_count  present_count  missing_percent
               notes            500              2            99.60
        loan_purpose            452             50            90.04
processing_timestamp            440             62            87.65
                 ssn              5            497             1.00
       date_of_birth              5            497             1.00
          ip_address              5            497             1.00
       annual_income              5            497             1.00
              gender              3            499             0.60
            zip_code              2            500             0.40
           full_name              0            502             0.00


In [21]:
governance_fields = [
    "consent_timestamp",
    "retention_until",
    "data_source",
    "processing_purpose"
]

results = []

for field in governance_fields:

    if field in audit_dataset.columns:
        missing = audit_dataset[field].isna().sum() + (audit_dataset[field] == "").sum()
    else:
        missing = len(audit_dataset)

    percent = (missing / len(audit_dataset)) * 100

    results.append([field, missing, round(percent,2)])

governance_df = pd.DataFrame(
    results,
    columns=["field","missing_count","missing_percent"]
).sort_values("missing_percent", ascending=False)

print(governance_df.to_string(index=False))

             field  missing_count  missing_percent
 consent_timestamp            502            100.0
   retention_until            502            100.0
       data_source            502            100.0
processing_purpose            502            100.0


The dataset lacks several governance fields required by the GDPR.

* No **consent_timestamp** field exists. This field records when an user gave consent to the processing of their data.

* The dataset is missing a **retention_until** field, which should define for how long data will remain stored in the system.

* The dataset has no **data_source** field  which should indicate where the personal data originated from.

* Furthermore there is no **processing_purpose** field. This field spicifies the resson for collecting and storing the data

### 4.4 Audit Query 4: Check Data Validity

Data Quality Dimension: Validity

In [24]:
# Check for numerical fields against logical ranges

#annual_inome >= 0
# credit_history_months >= 0
# debt_to_income >= 0
# savings_balance >= 0

# convert to numeric
audit_dataset["annual_income"] = pd.to_numeric(audit_dataset["annual_income"], errors="coerce")
audit_dataset["credit_history_months"] = pd.to_numeric(audit_dataset["credit_history_months"], errors="coerce")
audit_dataset["debt_to_income"] = pd.to_numeric(audit_dataset["debt_to_income"], errors="coerce")
audit_dataset["savings_balance"] = pd.to_numeric(audit_dataset["savings_balance"], errors="coerce")


# validity checks
invalid_income = (audit_dataset["annual_income"] < 0).sum()
invalid_credit_history = (audit_dataset["credit_history_months"] < 0).sum()
invalid_dti = (audit_dataset["debt_to_income"] <= 0).sum()
invalid_savings = (audit_dataset["savings_balance"] < 0).sum()

# results
validity_df = pd.DataFrame({
    "field": ["annual_income","credit_history_months","debt_to_income","savings_balance"],
    "invalid_cases": [invalid_income, invalid_credit_history, invalid_dti, invalid_savings]
})

print(validity_df)


                   field  invalid_cases
0          annual_income              0
1  credit_history_months              2
2         debt_to_income              0
3        savings_balance              1


Two fields contain values outside logical ranges:
 * 2 records show negative credit history months
 * 1 record contains a negative savings balance. These values indicate data entry or validation errors.

### 4.5 Audit Query 5: Detect Potential Bias

AI Act Requirment: Fairness Testing

In [ ]:
#standardize gender before fairness analysis

# create a copy to avoid modifying original
audit_dataset = audit_dataset.copy()

audit_dataset["gender"] = audit_dataset["gender"].replace({
    "M": "Male",
    "F": "Female",
    "male": "Male",
    "female": "Female"
})

# keep only Male and Female
audit_dataset = audit_dataset[audit_dataset["gender"].isin(["Male","Female"])]

# approval rate by gender
bias_df = (
    audit_dataset
    .groupby("gender")["loan_approved"]
    .agg(
        total="count",
        approved="sum"
    )
    .reset_index()
)

bias_df["approval_rate"] = bias_df["approved"] / bias_df["total"]

# approval rates
female_rate = bias_df.loc[bias_df["gender"] == "Female", "approval_rate"].values[0]
male_rate = bias_df.loc[bias_df["gender"] == "Male", "approval_rate"].values[0]

# disparate impact ratio
disparate_impact = min(female_rate, male_rate) / max(female_rate, male_rate)

print(bias_df.to_string(index=False))
print("Disparate Impact Ratio:", round(disparate_impact, 3))

gender  total  approved  approval_rate
Female    251       127       0.505976
  Male    248       163       0.657258
Disparate Impact Ratio: 0.77


### 4.6 Audit Query 6: Check Timeliness

Data quality Dimension: Timeliness

The data must be up to date and the timestamp must be useful.


In [35]:
stored = audit_dataset["processing_timestamp"].notna().sum()
missing = audit_dataset["processing_timestamp"].isna().sum()

pct_missing = missing / len(audit_dataset) * 100
pct_stored = stored / len(audit_dataset) * 100

print("Processing Timestamp Completeness:")
print(f"Stored: {stored}/{len(audit_dataset)} ({pct_stored:.1f}%)")
print(f"Missing: {missing}/{len(audit_dataset)} ({pct_missing:.1f}%)")

Processing Timestamp Completeness:
Stored: 62/499 (12.4%)
Missing: 437/499 (87.6%)


## 5. GDPR Article 5 Mapping 

###  Article 5(1)(a) - Lawfulness, fairness and transparency 

**Lawfulness:** The dataset doesn't contain any records of the legal basis under which the data was collected and processed. The audit confirmed that 


### Article 5(1)(b) - Purpose limitation 
No processing_purpose field exists in the dataset. Without this information the system cannot demonstrate that personal data is processed only for a specific, defined purpose.

### Article 5(1)(c) - Data minimisation 
Several personal identifiers are stored together, including SSN, full name, IP address and date of birth. Storing multiple direct identifiers increases re identification risk and may exceed what is necessary for loan decision processing.

### Article 5(1)(d) - Accuracy 
Data quality tests identified inconsistencies and invalid values. Examples include inconsistent gender labels and invalid financial values such as negative credit history months and negative savings balances.


### Article 5(1)(e) - Storage limitation 
The dataset lacks a retention_until field. Without a defined retention period the system cannot enforce limits on how long personal data remains stored.

### Article 5(1)(f) - Integrity and confidentiality 
Duplicate identifiers and inconsistent records weaken data integrity controls. Weak integrity controls increase risk of incorrect decisions and data misuse.

### 5.1 Lawful Basis

Under the GDPR, personal data must be processed under a valid lawful basis. This means an organization must have a legitimate reason for collecting and using personal data. NovaCred uses machine learning to make credit decisions, which requires access to applicants’ personal and financial information in order to train and operate its models.

In this context, the lawful basis for processing is the performance of a contract. Clients submit their personal and financial information when applying for a loan, and NovaCred processes this data to evaluate creditworthiness and decide whether the loan should be approved. Without processing this information, the company would not be able to assess applications or provide the requested financial service.



### 5.2  Data Minimization

The GDPR includes the principle of Data Minimization, which states that organizations should only collect and process the minimum amount of personal data necessary for their activities. If data is not required for their activity it should not be collected or stored.

In the case of NovaCred, the purpose of the data is to classify applicants as creditworthy or not. To check it, the company requires some financial and demographic information. However, several direct identifiers such as full name, email, IP address, and the internal system ID are not necessary for the functioning of the model and are still stored. For this reason, these fields were pseudonymized or removed to reduce the exposure of personal data.

### 5.3 Storage Limitation

The principle of Storage Limitation, states that data should only be stored for the purpose it was collected. Once it is no longer required, it should be deleted.

For NovaCred, personal and financial information, is stored to evaluate credit worthyness. Once a decision has been made regarding the approval or rejection of a loan, they no longer need to store all the initial data. For applicants, whose application was rejected, sotring their data won't be necessary after their process is closed. 

## 6. Privacy and Governance Gaps

### 6.1 Storage of Unnecessary Personal Identifiers

The dataset includes fields that are not relevant and do not contribute to the purpose of the credit evaluation process. The storage of individuals’ full name, email, SSN, and IP address is not necessary for the company’s credit assessment activity.

### 6.2 High Risk of Individual Identification from Quasi-Identifiers

This dataset contains attributes that can single out individuals even though they do not directly identify them. Variables such as Date of Birth, Zip Code, and gender act as quasi-identifiers. When combined, these attributes uniquely identify almost the entire set of individuals in the dataset.

### 6.3 No Data Retention Policy

There is no explicit policy indicating when the company plans to stop storing individuals’ data. The dataset also does not contain timestamps indicating when the data was first collected or stored. Without this, it becomes difficult to ensure that personal data is not stored longer than necessary.

### 6.4 No consent Tracking

The dataset does not contain any information regarding the authorization for the use of individuals’ data. NovaCred cannot show that individuals allowed them to collect and process their personal information. To address this issue, the dataset should include a field indicating whether consent was given, or a timestamp recording when consent was obtained.

### 6.5 No Audit Trail

The dataset contains a field indicating whether the loan was approved and a rejection reason explaining the outcome. However, the dataset does not include information such as when the decision was made, which model version generated the decision, or the risk score that led to the rejection.

### 6.6 No Human Oversight documention

NovaCred does not track who was responsible for each loan decision. The dataset does not include any fields indicating who made the decision, whether it was made automatically by the machine learning model or whether a human analyst reviewed the process. Without documentation of human oversight, it is not possible to ensure that automated credit decisions are monitored and corrected when necessary.

## 7. Governance Recommendations

### 7.1 Reduce Unnecessary Personal Identifiers

NovaCred should avoid storing personal data that are not relevant for the credit evaluation process. Fields such as full name, email, SSN, IP address, and system ID should be removed or replaced with pseudonymous identifiers. This reduces privacy risk and goes along with the principle of Data Minimization of the GDPR.

### 7.2 Reduce Identification Risk from Quasi Identifiers

There are variables like date of birth, zip code, and gender that create a high risk of identifying an individual when combined or even by themselves.  NovaCred should generalize or transform this fields to reduce risks. For example, replace the date of birth by age ranges, and zip codes by broader geographical regions.

### 7.3 Implement a Data Retention Policy

NovaCred should create a clear policy defining until when data is tored. Personal and Financial data should only be stored while the credit evaluation process is active. Once a decision has been made, no longer relevant information should be deleted, especially if the lon was rejected.

### 7.4 Add Consent Tracking

The company should record whether and when individuals gave them permission to colect and process their data. The dataset should include a field with a time stamp of when permission was given.

### 7.5 Improve the Decision Audit Trail

While the dataset includes a field indicating whether a loan was approved and a rejection_reason, this information alone is not sufficient to fully audit the decision process. The system should also monitor other fields such as the timestamp of the decision, the model version used, and the risk score produced by the model. These records would allow the company to explain and verify how each credit decision was generated.

### 7.6 Implement Document Human Overview

The dataset does not contain any information indicating whether the decision was reviewed by a human analyst. NovaCred should add a field identifying the employee responsible for reviewing the decision, a confirmation record indicating that the analyst validated or corrected the model’s output, and a timestamp showing when the review occurred.